## Example 1: S3Client Type Annotations for Bucket and Object Operations

In [ ]:
from types_aiobotocore_s3 import S3Client
from types_aiobotocore_s3.type_defs import ListBucketsOutputTypeDef, BucketTypeDef, ListObjectsV2OutputTypeDef
from aiobotocore.session import get_session
from botocore.exceptions import NoCredentialsError

async def demonstrate_s3_client_types():
    session = get_session()
    async with session.create_client("s3", region_name="us-east-1") as client:
        client: S3Client  # Type annotation enables IDE autocomplete
        try:
            buckets_response: ListBucketsOutputTypeDef = await client.list_buckets()
            print(f"S3 Buckets: {len(buckets_response.get('Buckets', []))}")
            for bucket in buckets_response.get("Buckets", [])[:3]:
                bucket_typed: BucketTypeDef = bucket
                print(f"  Bucket: {bucket_typed.get('Name')} (Created: {bucket_typed.get('CreationDate')})")
            if buckets_response.get('Buckets'):
                bucket_name = buckets_response['Buckets'][0]['Name']
                objects_response: ListObjectsV2OutputTypeDef = await client.list_objects_v2(Bucket=bucket_name, MaxKeys=5)
                print(f"\nObjects in {bucket_name}: {objects_response.get('KeyCount', 0)}")
        except NoCredentialsError:
            print("No AWS credentials (expected)")
        except Exception as e:
            print(f"No buckets/access denied (expected): {type(e).__name__}")
        print("S3Client provides 100+ typed methods for S3 operations")

await demonstrate_s3_client_types()

## Example 2: Using Paginator Types for S3 Resources

In [ ]:
from types_aiobotocore_s3 import S3Client
from types_aiobotocore_s3.paginator import ListObjectsV2Paginator, ListMultipartUploadsPaginator, ListPartsPaginator
from aiobotocore.session import get_session
from botocore.exceptions import ClientError, NoCredentialsError

async def demonstrate_s3_paginator_types():
    session = get_session()
    async with session.create_client("s3", region_name="us-east-1") as client:
        client: S3Client
        objects_paginator: ListObjectsV2Paginator = client.get_paginator("list_objects_v2")
        uploads_paginator: ListMultipartUploadsPaginator = client.get_paginator("list_multipart_uploads")
        parts_paginator: ListPartsPaginator = client.get_paginator("list_parts")
        print("S3 Paginator types:")
        print(f"  ListObjectsV2Paginator: {type(objects_paginator).__name__}")
        print(f"  ListMultipartUploadsPaginator: {type(uploads_paginator).__name__}")
        print(f"  ListPartsPaginator: {type(parts_paginator).__name__}")
        try:
            buckets = await client.list_buckets()
            if buckets.get('Buckets'):
                bucket_name = buckets['Buckets'][0]['Name']
                object_count = 0
                async for page in objects_paginator.paginate(Bucket=bucket_name, PaginationConfig={'MaxItems': 50}):
                    object_count += page.get('KeyCount', 0)
                print(f"Found {object_count} objects in {bucket_name} (first 50 at most)")
        except (NoCredentialsError, ClientError) as e:
            print(f"No credentials or access denied (expected): {type(e).__name__}")
        print("Package provides 10+ S3 paginator types")

await demonstrate_s3_paginator_types()

## Example 3: Using TypeDefs for Bucket Creation and Configuration

In [ ]:
from types_aiobotocore_s3.type_defs import CreateBucketConfigurationTypeDef, CreateBucketRequestTypeDef, PutBucketVersioningRequestTypeDef, VersioningConfigurationTypeDef, TagTypeDef, TaggingTypeDef
from types_aiobotocore_s3 import S3Client
from types_aiobotocore_s3.type_defs import BucketTypeDef, ListBucketsOutputTypeDef, ListObjectsV2OutputTypeDef
from types_aiobotocore_s3.paginator import ListMultipartUploadsPaginator, ListObjectsV2Paginator, ListPartsPaginator

def demonstrate_bucket_creation_typedefs():
    bucket_config: CreateBucketConfigurationTypeDef = {"LocationConstraint": "us-west-2"}
    create_bucket: CreateBucketRequestTypeDef = {
        "Bucket": "my-app-data-bucket",
        "CreateBucketConfiguration": bucket_config,
        "ACL": "private"
    }
    print("Type-safe Bucket Creation Request:")
    print(f"  Bucket Name: {create_bucket['Bucket']}")
    print(f"  Region: {bucket_config['LocationConstraint']}")
    print(f"  ACL: {create_bucket['ACL']}")
    versioning_config: VersioningConfigurationTypeDef = {"Status": "Enabled", "MFADelete": "Disabled"}
    versioning_request: PutBucketVersioningRequestTypeDef = {
        "Bucket": create_bucket['Bucket'],
        "VersioningConfiguration": versioning_config
    }
    tags: list[TagTypeDef] = [{"Key": "Environment", "Value": "Production"}, {"Key": "CostCenter", "Value": "Engineering"}]
    tagging: TaggingTypeDef = {"TagSet": tags}
    print(f"\nVersioning: {versioning_config['Status']}")
    print(f"Tags: {len(tags)} tags configured")
    for tag in tags:
        print(f"  {tag['Key']}={tag['Value']}")
    print("TypeDefs ensure type safety for S3 bucket management")

demonstrate_bucket_creation_typedefs()

## Example 4: Using TypeDefs for Object Operations

In [ ]:
from types_aiobotocore_s3.type_defs import PutObjectRequestTypeDef, CopyObjectRequestTypeDef, GetObjectRequestTypeDef, DeleteObjectRequestTypeDef, ObjectTypeDef

def demonstrate_object_operations_typedefs():
    put_request: PutObjectRequestTypeDef = {
        "Bucket": "my-app-data-bucket",
        "Key": "data/user-123/profile.json",
        "Body": b'{"name": "John Doe", "email": "john@example.com"}',
        "ContentType": "application/json",
        "ServerSideEncryption": "AES256",
        "Metadata": {"userId": "123", "version": "1"}
    }
    print("Type-safe Put Object Request:")
    print(f"  Bucket: {put_request['Bucket']}")
    print(f"  Key: {put_request['Key']}")
    print(f"  Content-Type: {put_request['ContentType']}")
    print(f"  Encryption: {put_request['ServerSideEncryption']}")
    copy_request: CopyObjectRequestTypeDef = {
        "Bucket": "my-app-backups",
        "Key": "backup/profile.json",
        "CopySource": f"{put_request['Bucket']}/{put_request['Key']}"
    }
    get_request: GetObjectRequestTypeDef = {"Bucket": put_request['Bucket'], "Key": put_request['Key']}
    delete_request: DeleteObjectRequestTypeDef = {"Bucket": put_request['Bucket'], "Key": put_request['Key']}
    print(f"\nCopy Source: {copy_request['CopySource']}")
    print(f"Copy Destination: {copy_request['Bucket']}/{copy_request['Key']}")
    print("TypeDefs ensure type safety for S3 object operations")

demonstrate_object_operations_typedefs()

## Example 5: Using TypeDefs for Multipart Uploads

In [ ]:
from types_aiobotocore_s3.type_defs import CreateMultipartUploadRequestTypeDef, UploadPartRequestTypeDef, CompletedPartTypeDef, CompletedMultipartUploadTypeDef, CompleteMultipartUploadRequestTypeDef

def demonstrate_multipart_upload_typedefs():
    create_upload: CreateMultipartUploadRequestTypeDef = {
        "Bucket": "my-large-files-bucket",
        "Key": "videos/large-video.mp4",
        "ContentType": "video/mp4",
        "ServerSideEncryption": "AES256"
    }
    print("Type-safe Multipart Upload Initialization:")
    print(f"  Bucket: {create_upload['Bucket']}")
    print(f"  Key: {create_upload['Key']}")
    print(f"  Content-Type: {create_upload['ContentType']}")
    upload_id = "EXAMPLE-UPLOAD-ID-ABC123"
    upload_part: UploadPartRequestTypeDef = {
        "Bucket": create_upload['Bucket'],
        "Key": create_upload['Key'],
        "PartNumber": 1,
        "UploadId": upload_id,
        "Body": b"video_chunk_data_here"
    }
    completed_parts: list[CompletedPartTypeDef] = [{"ETag": "etag-part1", "PartNumber": 1}, {"ETag": "etag-part2", "PartNumber": 2}]
    completed_upload: CompletedMultipartUploadTypeDef = {"Parts": completed_parts}
    complete_request: CompleteMultipartUploadRequestTypeDef = {"Bucket": create_upload['Bucket'], "Key": create_upload['Key'], "UploadId": upload_id, "MultipartUpload": completed_upload}
    print(f"\nMultipart Upload: {len(completed_parts)} parts completed")
    print("TypeDefs ensure type safety for S3 multipart upload operations")

demonstrate_multipart_upload_typedefs()